In [ ]:
import functools
import json

import eradiate
import numpy as np
import xarray as xr

from eradiate.units import unit_registry as ureg

eradiate.set_mode("mono")

In [ ]:
# Reindex land cover data
lc_map = xr.load_dataset("data/landcover_beijing.nc")["landcover_class"]
lc_values = np.unique(lc_map)

lc_indexes = xr.zeros_like(lc_map, dtype="int64")
lc_indexes.attrs = {}

for i, v in enumerate(lc_values):
    lc_indexes.values[lc_map == v] = i

lc_indexes.plot.imshow()

In [ ]:
# Declare materials
with open("data/colormap.json") as f:
    colormap = json.load(f)

colormap_indexes = {
    i: colormap[str(int(v))] for i, v in enumerate(lc_values)
}
display(colormap_indexes)

def material_reflectance(mat_id, w):
    mat_rgb = colormap_indexes[mat_id]

    if w == 440.0:
        return float(mat_rgb[2]) / 255
    elif w == 550.0:
        return float(mat_rgb[1]) / 255
    elif w == 660.0:
        return float(mat_rgb[0]) / 255
    else:
        raise ValueError

reflectances = {
    i: functools.partial(material_reflectance, i) for i in range(len(lc_values))
}
reflectances

In [ ]:
%reload_ext eradiate.notebook

# Generate Eradiate scene
kdict = {
    "elevation": {
        "type": "ply",
        "filename": "data/dem_beijing.ply",
        "face_normals": False,
    }
}

exp = eradiate.experiments.AtmosphereExperiment(
    atmosphere=None,
    surface=None,
    measures={
        "type": "perspective",
        "origin": [-1000, -1000, 1000] * ureg.km,
        "target": [0, 0, 0],
        "up": [0, 0, 1],
        "fov": 5,
        "film_resolution": (320, 240),
        "srf": {"type": "delta", "wavelengths": [440, 550, 660]},
    },
    illumination={
        "type": "directional",
        "zenith": 30.0,
    },
    kdict=kdict,
)
result = eradiate.run(exp)

In [ ]:
import matplotlib.pyplot as plt

da = result["radiance"].squeeze().transpose("y_index", "x_index", "w")
# display(da)
plt.imshow((da.values * 1.8) ** (1.0 / 2.2))
plt.axis("off")